In [ ]:
import os
import glob
import sys 
sys.path.append('../..')

from pathlib import Path

from zipfile import ZipFile
import shutil

import astropy.io.fits as fits
import sunpy

from utilities.file_formats import *
from utilities.db_queries import *

from tqdm.notebook import tqdm
import json

import re
from copy import deepcopy

# Make Dataset

In [ ]:
#Path to the sqlite database:
sqlite_path = "../../datasets/WL2CAL/uset_drawings_2023.sqlite"

# Path to the Traslation dataset:
dataset_dir = Path('../datasets/WL2CAL/')


wl_root_dir = dataset_dir / 'source_WL-UPTO2023'
cal_root_dir = dataset_dir / 'source_CallK-UPTO2023'
dr_root_dir = dataset_dir / 'source_DR-UPTO2023'

drawings_origin = Path('../datasets/ImageTranslation_dataset/drawings/validated_drawings')


In [3]:
wl_images = {}
# # list all the files in all the zip files in both directories

wl_files = sorted(glob.glob(str(wl_root_dir / '**/*.FTS'), recursive=True))

images_in_zip = [item for item in sorted(wl_files) if item[-4:] == '.FTS' ]
images_bn_in_zip = [os.path.basename(item) for item in images_in_zip]
fail_indexes = [i for i, item in enumerate(images_bn_in_zip) if  (('failure' in item) or not item.startswith('UPH'))]
images_bn_in_zip = [item for i,item in enumerate(images_bn_in_zip) if i not in fail_indexes]
images_in_zip = [item for i,item in enumerate(images_in_zip) if i not in fail_indexes]

images_datetime_in_zip = [whitelight_to_datetime(item.split('_')[0]) for item in images_bn_in_zip]


wl_images = {'images_in_zip': images_in_zip,
                            'images_bn_in_zip': images_bn_in_zip,
                            'images_datetime_in_zip': images_datetime_in_zip
                            }

print(wl_images['images_bn_in_zip'][:3])


cal_images = {}
# # list all the files in all the zip files in both directories
cal_files = sorted(glob.glob(str(cal_root_dir / '**/*.FTS'), recursive=True))

images_in_zip = [item for item in sorted(cal_files) if item[-4:] == '.FTS' ]
images_bn_in_zip = [os.path.basename(item) for item in images_in_zip]
fail_indexes = [i for i, item in enumerate(images_bn_in_zip) if (('failure' in item) or not item.startswith('UCC'))]
images_bn_in_zip = [item for i,item in enumerate(images_bn_in_zip) if i not in fail_indexes]
images_in_zip = [item for i,item in enumerate(images_in_zip) if i not in fail_indexes]


images_datetime_in_zip = [calcium_to_datetime(item.split('_')[0]) for item in images_bn_in_zip]


cal_images = {'images_in_zip': images_in_zip,
                            'images_bn_in_zip': images_bn_in_zip,
                            'images_datetime_in_zip': images_datetime_in_zip
                            }

print(cal_images['images_bn_in_zip'][:3])



['UPH20120104095758.FTS', 'UPH20120104121430.FTS', 'UPH20120105120501.FTS']
['UCC20120711081726.FTS', 'UCC20120711082307.FTS', 'UCC20120711082941.FTS']


In [4]:
def per_day_dict(dt_dict):
    # From all the datetime dicts, make a dict with 3 layers of keys: year, month, day 
    # and the values are lists of times, which are the concateated values of the
    #  "hours", "minutes" and "seconds" keys
    per_day = {}
    for dt in dt_dict:
        year = dt['year']
        month = dt['month']
        day = dt['day']
        if year not in per_day:
            per_day[year] = {}
        if month not in per_day[year]:
            per_day[year][month] = {}
        if day not in per_day[year][month]:
            per_day[year][month][day] = []

        per_day[year][month][day].append(f"{dt['hours']}:{dt['minutes']}:{dt['seconds']}")

    return per_day

def per_day_dict_with_fn(dt_dict_lst, fn_lst):
    # From all the datetime dicts, make a dict with 3 layers of keys: year, month, day 
    # and the values are lists of times, which are the concateated values of the
    #  "hours", "minutes" and "seconds" keys
    per_day = {}
    for dt, fn in list(zip(dt_dict_lst, fn_lst)):
        year = dt['year']
        month = dt['month']
        day = dt['day']
        if year not in per_day:
            # print(year)
            per_day[year] = {}
        if month not in per_day[year]:
            # print(month)
            per_day[year][month] = {}
        if day not in per_day[year][month]:
            # print(day)
            per_day[year][month][day] = []
        
        entry = {'fn': fn, 'time': f"{dt['hours']}:{dt['minutes']}:{dt['seconds']}"}
        per_day[year][month][day].append(entry)
        # print(per_day)

    return deepcopy(per_day)

In [5]:
# res = query_datetimes_between_sqlite(sqlite_path, "2013-01-01 00:00:00", "2015-12-31 23:59:59", "drawings")
res = query_datetimes_between_sqlite(sqlite_path, "2012-01-01 00:00:00", "2023-12-31 23:59:59", "drawings")

res_fn = [item['Filename'] for item in res]
res_dt = [item['DateTime'] for item in res]

print(len(res_dt))
res_datetime = [db_string_to_datetime(item) for item in res_dt]
# print(res_datetime)
print(min(res_dt))

# per_day_dr = per_day_dict(res_datetime)
per_day_dr = per_day_dict_with_fn(res_datetime, res_fn)
print(per_day_dr)
print(per_day_dr["2019"]["02"].keys())

3264
2012-01-04 09:40:00
{'2012': {'01': {'04': [{'fn': 'usd201201040940.jpg', 'time': '09:40:00'}], '05': [{'fn': 'usd201201051200.jpg', 'time': '12:00:00'}], '08': [{'fn': 'usd201201081000.jpg', 'time': '10:00:00'}], '11': [{'fn': 'usd201201110910.jpg', 'time': '09:10:00'}], '13': [{'fn': 'usd201201131110.jpg', 'time': '11:10:00'}], '14': [{'fn': 'usd201201141015.jpg', 'time': '10:15:00'}], '15': [{'fn': 'usd201201151000.jpg', 'time': '10:00:00'}], '16': [{'fn': 'usd201201160920.jpg', 'time': '09:20:00'}], '17': [{'fn': 'usd201201170945.jpg', 'time': '09:45:00'}, {'fn': 'usd201201171400.jpg', 'time': '14:00:00'}], '23': [{'fn': 'usd201201231100.jpg', 'time': '11:00:00'}], '27': [{'fn': 'usd201201270940.jpg', 'time': '09:40:00'}, {'fn': 'usd201201271135.jpg', 'time': '11:35:00'}], '31': [{'fn': 'usd201201311135.jpg', 'time': '11:35:00'}]}, '02': {'01': [{'fn': 'usd201202010920.jpg', 'time': '09:20:00'}, {'fn': 'usd201202011150.jpg', 'time': '11:50:00'}], '02': [{'fn': 'usd201202020905

In [ ]:
per_day_wl = {}
per_day_cal = {}

per_day_wl_bn = {}
per_day_cal_bn = {}



# Get the list of images
wl_list = wl_images['images_bn_in_zip']
cal_list = cal_images['images_bn_in_zip']

wl_datetimes = wl_images['images_datetime_in_zip']
cal_datetimes = cal_images['images_datetime_in_zip']

print(wl_datetimes[:3])
print(cal_datetimes[:3])
print(wl_list[:3])
print(cal_list[:3])

cur_per_day_wl = per_day_dict_with_fn(wl_datetimes, wl_list)
cur_per_day_cal = per_day_dict_with_fn(cal_datetimes, cal_list)


per_day_wl.update(cur_per_day_wl)
per_day_cal.update(cur_per_day_cal)

print(per_day_wl.keys())
print(per_day_cal.keys())
print(per_day_dr.keys())

per_day_dr['2013'].keys()

[{'year': '2012', 'month': '01', 'day': '04', 'hours': '09', 'minutes': '57', 'seconds': '58'}, {'year': '2012', 'month': '01', 'day': '04', 'hours': '12', 'minutes': '14', 'seconds': '30'}, {'year': '2012', 'month': '01', 'day': '05', 'hours': '12', 'minutes': '05', 'seconds': '01'}]
[{'year': '2012', 'month': '07', 'day': '11', 'hours': '08', 'minutes': '17', 'seconds': '26'}, {'year': '2012', 'month': '07', 'day': '11', 'hours': '08', 'minutes': '23', 'seconds': '07'}, {'year': '2012', 'month': '07', 'day': '11', 'hours': '08', 'minutes': '29', 'seconds': '41'}]
['UPH20120104095758.FTS', 'UPH20120104121430.FTS', 'UPH20120105120501.FTS']
['UCC20120711081726.FTS', 'UCC20120711082307.FTS', 'UCC20120711082941.FTS']
dict_keys(['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'])
dict_keys(['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'])
dict_keys(['2012', '2013', '2014', '2015', '2016', '2017', '

dict_keys(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'])

In [ ]:
from datetime import datetime, timedelta

def get_time_difference(time1, time2):
    time_format = '%H:%M:%S'
    
    # Parse the time strings into datetime objects
    time1_obj = datetime.strptime(time1, time_format)
    time2_obj = datetime.strptime(time2, time_format)
    
    # Calculate the time difference
    time_difference = time1_obj - time2_obj
    
    return time_difference


def is_time_difference_less_than_threshold(time1, time2, threshold):    
    # Calculate the time difference
    time_difference = abs(get_time_difference(time1, time2))
    
    # Compare the time difference with the threshold
    return time_difference < threshold

def contains_quality(input_string, lower_better=True):
    # Define the regular expression pattern with capturing group
    pattern = r'q([0-9])'

    # Use the re.search() function to find a match
    match = re.search(pattern, input_string)

    # Check if a match is found
    if match:
        x_value = match.group(1)
        # print("Found a match: 'q' followed by", x_value)
        return True, int(x_value)
    else:
        # print("No match found.")
        if 'SYNOPTIC' in input_string:
            if lower_better:
                return True, 0
            else:
                return True, 10 # best if highest value seeked
        return False, -1


def get_best_match_idx(dr_time, date ,candidates, zip_dir, zip_file, lower_better=False, timediff_threshold=timedelta(hours=4, minutes=0)):
    y,m,d = date

    # create temprary folder
    tmp_dir = Path('./tmp')
    if not tmp_dir.exists():
        tmp_dir.mkdir()

    created_files = []
    # extract all images in the candidate list to the temporary folder
    with ZipFile(f"{zip_dir}/{zip_file}", 'r') as wl_zipf:
        # print(f"ICI:  {zip_dir}/{zip_file}")
        for candidate in candidates:
            my_str = f'{zip_file.split(".")[0]}/{y}/{m}/{candidate["fn"]}' if  "calcium" in zip_file else f'{y}/{m}/{candidate["fn"]}'
            source_wl = wl_zipf.open(my_str)
            target_wl = open(f"{tmp_dir}/{candidate['fn']}", "wb")
            

            with source_wl, target_wl:
                created_files.append( tmp_dir / candidate['fn'])

                # # if target file exists, skip this candidate
                # if not os.path.exists(f"{tmp_dir}/{candidate['fn']}"):
                shutil.copyfileobj(source_wl, target_wl)

    ######
    # GET THE QUALITY OF EACH IMAGE
    ######
    # open all the images in the temporary folder 
    wl_qualities = []
    for c in created_files:
        #open the FTS file and read the header
        try:
            hdulst_wl:fits.HDUList = fits.open(c)
        except OSError as e: # this is a corrupted file
            print ("Error: %s - %s." % (e.filename, e.strerror))
            return -1, -1, -1
        test_img_wl = hdulst_wl[0]
        header_wl = test_img_wl.header

        ######## TEST ########
        if 'OBS_MODE' in header_wl:
            is_q, qv = contains_quality(header_wl['OBS_MODE'], lower_better=lower_better)
            wl_qualities.append(qv)
        else:
            is_q, qv = contains_quality(str(header_wl['COMMENT']), lower_better=lower_better)
            wl_qualities.append(qv)

    # delete the temporary folder, and all the files in it
    try:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    except OSError as e:
        print ("Error: %s - %s." % (e.filename, e.strerror))
        return -1

    ######
    # GET THE TIME DIFFERENCE OF EACH IMAGE
    ######
    times = [item['time'] for item in candidates]
    times_readable = [str(item).split('.')[0] for item in times]
    # print(f"TIMES Readable: {times_readable}")
    # time_differences = [get_time_difference(dr_time, item) for item in times]
    time_differences = [abs(get_time_difference(dr_time, item)) for item in times]
    # make it readable in hours, minutes, seconds
    time_difference_readable = [str(item).split('.')[0] for item in time_differences]
    # print(f"TIME DIFFERENCES: {time_differences}")
    # print(f"TIME DIFFERENCES READABLE: {time_difference_readable}")

    ######
    # GET THE INDEX OF THE BEST MATCH
    ######
    # The best match is the one with with highest quality (lowest number) and the lowest time difference
    # Two images do not have the same time difference
    # If there are multiple images with the same time difference and quality, the first one is selected
    # If there are no images with quality, the first one is selected
    # Also, make sure that the time difference is less than the threshold
    best_match_idx = -1
    best_match_time_diff = timediff_threshold
    best_match_quality = 0
    for i, (time_diff, quality) in enumerate(zip(time_differences, wl_qualities)):
        if (quality >= best_match_quality):
            if (time_diff < timediff_threshold):
                if (time_diff < best_match_time_diff):
                    best_match_idx = i
                    best_match_time_diff = time_diff
                    best_match_quality = quality
                else: 
                    if (quality > best_match_quality):
                        best_match_idx = i
                        best_match_time_diff = time_diff
                        best_match_quality = quality
    
    if best_match_idx == -1:
        # print("No best match found")
        return -1, -1, -1
    else:
        return best_match_idx, best_match_time_diff, best_match_quality


def get_best_match_idx_nozip(dr_time, date ,candidates, candatates_folder, lower_better=False, timediff_threshold=timedelta(hours=4, minutes=0)):
    y,m,d = date

    candidates_files = [ candatates_folder / y / m / item['fn'] for item in candidates ]

    ######
    # GET THE QUALITY OF EACH IMAGE
    ######
    # open all the images in the temporary folder 
    wl_qualities = []
    for c in candidates_files:
        #open the FTS file and read the header
        try:
            hdulst_wl:fits.HDUList = fits.open(c)
        except OSError as e: # this is a corrupted file
            print ("Error: %s - %s." % (e.filename, e.strerror))
            return -1, -1, -1
        
        test_img_wl = hdulst_wl[0]
        header_wl = test_img_wl.header
        ######## TEST ########
        if 'OBS_MODE' in header_wl:
            is_q, qv = contains_quality(header_wl['OBS_MODE'], lower_better=lower_better)
            wl_qualities.append(qv)
        else:
            is_q, qv = contains_quality(str(header_wl['COMMENT']), lower_better=lower_better)
            wl_qualities.append(qv)

    ######
    # GET THE TIME DIFFERENCE OF EACH IMAGE
    ######
    times = [item['time'] for item in candidates]
    times_readable = [str(item).split('.')[0] for item in times]
    # print(f"TIMES Readable: {times_readable}")
    # time_differences = [get_time_difference(dr_time, item) for item in times]
    time_differences = [abs(get_time_difference(dr_time, item)) for item in times]
    # make it readable in hours, minutes, seconds
    time_difference_readable = [str(item).split('.')[0] for item in time_differences]
    # print(f"TIME DIFFERENCES: {time_differences}")
    # print(f"TIME DIFFERENCES READABLE: {time_difference_readable}")

    ######
    # GET THE INDEX OF THE BEST MATCH
    ######
    # The best match is the one with with highest quality (highest number) and the lowest time difference
    # Two images do not have the same time difference
    # If there are multiple images with the same time difference and quality, the first one is selected
    # If there are no images with quality, the first one is selected
    # Also, make sure that the time difference is less than the threshold
    best_match_idx = -1
    best_match_time_diff = timediff_threshold
    best_match_quality = 0
    for i, (time_diff, quality) in enumerate(zip(time_differences, wl_qualities)):
        if (quality >= best_match_quality):
            if (time_diff < timediff_threshold):
                if (time_diff < best_match_time_diff):
                    best_match_idx = i
                    best_match_time_diff = time_diff
                    best_match_quality = quality
                else: 
                    if (quality > best_match_quality):
                        best_match_idx = i
                        best_match_time_diff = time_diff
                        best_match_quality = quality
            # if (time_diff < best_match_time_diff) and (time_diff < timediff_threshold):
            #     best_match_idx = i
            #     best_match_time_diff = time_diff
            #     best_match_quality = quality
    
    # print(f"best_match_idx: {best_match_idx}, best_match_time_diff: {best_match_time_diff}, best_match_quality: {best_match_quality}")
    
    if best_match_idx == -1:
        # print("No best match found")
        return -1, -1, -1
    else:
        return best_match_idx, best_match_time_diff, best_match_quality


# Get DR - WL - CaIIK triplets


In [ ]:
# multiprocess version
def process_one_day(args):
    # print(args)
    inp_idx, year_dr, month_dr, day_dr, \
    per_day_dr, per_day_wl, per_day_cal,\
    wl_root_dir, cal_root_dir, \
    dr_dir, wl_dir, cal_dir, \
    sqlite_path, lower_better, max_delta = args

    # make sure there is a corresponding entry in the other two dictionaries
    # if not, skip this day
    try:
        assert (year_dr in per_day_wl) and (year_dr in per_day_cal)
        assert (month_dr in per_day_wl[year_dr]) and (month_dr in per_day_cal[year_dr])
        assert (day_dr in per_day_wl[year_dr][month_dr]) and (day_dr in per_day_cal[year_dr][month_dr])
    except AssertionError:
        return -1, None
        
    
    # MAYBE ADD A BETTER WAY TO SELECT IDX (MAYBE NOT, the number of timediffs too large is very small)
    idx_dr = 0

    cur_fn_dr = per_day_dr[year_dr][month_dr][day_dr][idx_dr]["fn"]
    cur_time_dr = per_day_dr[year_dr][month_dr][day_dr][idx_dr]["time"]


    if not len(per_day_wl[year_dr][month_dr][day_dr]) > 1:
        return -1, None
    
    idx_wl, td_wl, q_wl = get_best_match_idx_nozip(
                                    per_day_dr[year_dr][month_dr][day_dr][idx_dr]["time"], 
                                    (year_dr, month_dr, day_dr),
                                    per_day_wl[year_dr][month_dr][day_dr],
                                    wl_root_dir,
                                    lower_better=lower_better,
                                    timediff_threshold=max_delta,
                                ) 
    idx_cal, td_cal, q_cal = get_best_match_idx_nozip(
                                    per_day_dr[year_dr][month_dr][day_dr][idx_dr]["time"], 
                                    (year_dr, month_dr, day_dr),
                                    per_day_cal[year_dr][month_dr][day_dr],
                                    cal_root_dir,
                                    lower_better=lower_better,
                                    timediff_threshold=max_delta,
                                    )

    cur_fn_wl = per_day_wl[year_dr][month_dr][day_dr][idx_wl]["fn"]
    cur_fn_cal = per_day_cal[year_dr][month_dr][day_dr][idx_cal]["fn"]

    cur_time_wl = per_day_wl[year_dr][month_dr][day_dr][idx_wl]["time"]
    cur_time_cal = per_day_cal[year_dr][month_dr][day_dr][idx_cal]["time"]

    # Make sure that the time deltas are less than 4 hours
    wl_ok = is_time_difference_less_than_threshold(cur_time_dr, cur_time_wl, max_delta)
    cal_ok = is_time_difference_less_than_threshold(cur_time_dr, cur_time_cal, max_delta)
    
    if wl_ok and cal_ok:               
        ###############################
        # extract the calibration data
        db_string = f'{year_dr}-{month_dr}-{day_dr} {cur_time_dr}'
        res_dr_calib = query_table_sqlite(sqlite_path, db_string, 'calibrations')
        # if there is no calibration data, skip this day
        if len(res_dr_calib) == 0:
            return -1, None

        res_dr_calib[0].pop('id', None)

        active_regions_db = query_table_sqlite(sqlite_path, db_string, 'sGroups')
        
        active_regions = [{
            'Latitude': float(item['Latitude']),
            'Longitude': float(item['Longitude']),
        } for item in active_regions_db]

        ###############################
        # extract the wl and cal images 
        source_wl = wl_root_dir / year_dr / month_dr / cur_fn_wl
        source_cal = cal_root_dir / year_dr / month_dr / cur_fn_cal

        target_wl = wl_dir / cur_fn_wl
        target_cal = cal_dir / cur_fn_cal

        # copy the files
        if not target_wl.exists():
            shutil.copyfile(source_wl, target_wl)
        if not target_cal.exists():
            shutil.copyfile(source_cal, target_cal)

        wl_name = cur_fn_wl
        cal_name = cur_fn_cal
        dr_name = cur_fn_dr
        
        ######### Obtain the time differences between the images
        dr_datetime = drawing_name_to_datetime(dr_name)
        dr_datetime['seconds'] = '00'
        wl_datetime = whitelight_to_datetime(wl_name)
        cal_datetime = calcium_to_datetime(cal_name)

        dr_time =  f"{dr_datetime['hours']}:{dr_datetime['minutes']}:{dr_datetime['seconds']}"
        wl_time =  f"{wl_datetime['hours']}:{wl_datetime['minutes']}:{wl_datetime['seconds']}"
        cal_time =  f"{cal_datetime['hours']}:{cal_datetime['minutes']}:{cal_datetime['seconds']}"

        td_wl2cal = get_time_difference(wl_time, cal_time)
        td_dr2wl = get_time_difference(dr_time, wl_time)
        ######### Generate the entry
        entry = {
            "dr_name": dr_name ,
            "wl_name":  wl_name,
            "cal_name":  cal_name,
            "dr_datetime": db_string_to_datetime(db_string),
            "dr_calib": res_dr_calib[0],
            "active_regions": active_regions,

            'wl2cal_time_diff': td_wl2cal.total_seconds(),
            "dr2wl_time_diff": td_dr2wl.total_seconds(),
        }
        return 0, entry
    else:
        print(f"Time difference too large: {cur_time_dr} - {cur_time_wl} - {cur_time_cal}")
        return 1, None
    

In [ ]:
from multiprocessing import Pool
from itertools import repeat
import time


lower_better = False  # Defines the preference for the quality value -> if True Synoptic = quality 0 else quality 10
max_delta = timedelta(hours=4, minutes=0)  # 4 hours

wl_dir = dataset_dir / "paired_wl2cal_whitelight_4h"
cal_dir = dataset_dir / "paired_wl2cal_calcium_4h"
dr_dir = dataset_dir / "paired_wl2cal_drawing_4h"

if not wl_dir.exists():
    wl_dir.mkdir()
if not cal_dir.exists():
    cal_dir.mkdir()
if not dr_dir.exists():
    dr_dir.mkdir()

extract_from_zip = False
extract_drawing = False

triplets = []

num_missing = 0
num_time_diff_too_large = 0

wrote = False
counter = 0

num_processes = 48

for year_dr, v_year_dr in tqdm(per_day_dr.items()):

    for month_dr, v_month_dr in tqdm(v_year_dr.items(), desc="Month", leave=False):

        # for day_dr, v_day_dr in tqdm(v_month_dr.items(), desc="Day", leave=False):
        num_days = len(v_month_dr)
         
        with Pool(num_processes) as p:
            results = list(tqdm(p.imap(process_one_day, zip (range(num_days), 
                                                   repeat(year_dr), repeat(month_dr),
                                                     list(v_month_dr.keys()), 
                                                     repeat(per_day_dr), repeat(per_day_wl), repeat(per_day_cal), 
                                                     repeat(wl_root_dir), repeat(cal_root_dir), 
                                                     repeat(dr_dir), repeat(wl_dir), repeat(cal_dir), 
                                                     repeat(sqlite_path), repeat(lower_better), repeat(max_delta))), total=num_days, leave=False))
            for res in results:
                if res[0] == 0:
                    triplets.append(res[1])
                elif res[0] == -1: # missing day
                    num_missing += 1
                else:
                    num_time_diff_too_large += 1
                    
        print(f"finished month {month_dr}")
    print(f"finished year {year_dr}")


print(f"{len(triplets)} triplets")
print(f"{num_missing} missing days")
print(f"{num_time_diff_too_large} time differences too large")
             

  0%|          | 0/12 [00:00<?, ?it/s]

Month:   0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

...

  0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

2421 triplets
659 missing days
29 time differences too large


In [ ]:
with open(dataset_dir / 'wl_dr_cal_matchings_new_max4h_aligned.json', 'w') as fp:
    json.dump(triplets, fp, indent=4)

TypeError: Object of type timedelta is not JSON serializable

In [ ]:
# for each triplet, extract the files
for triplet in triplets[:1]:
    print(triplet)
    
print(len(triplets))

{'dr_name': 'usd201207110845.jpg', 'wl_name': 'UPH20120711081533.FTS', 'cal_name': 'UCC20120711082941.FTS', 'dr_datetime': {'year': '2012', 'month': '07', 'day': '11', 'hours': '08', 'minutes': '45', 'seconds': '00'}, 'dr_calib': {'DateTime': '2012-07-11 08:45:00', 'NorthX': 826.0, 'NorthY': 183.0, 'CenterX': 826.0, 'CenterY': 922.0, 'Radius': 739.0, 'AngleScan': 0.0}, 'active_regions': [{'Latitude': 0.181589, 'Longitude': 2.002292}, {'Latitude': -0.247774, 'Longitude': 1.868894}, {'Latitude': -0.360601, 'Longitude': 1.685919}, {'Latitude': -0.264275, 'Longitude': 1.489305}]}


In [20]:
new_triplets = deepcopy(triplets)
for i, t in enumerate(new_triplets):
    td_wl2cal = new_triplets[i]["wl2cal_time_diff"]
    new_triplets[i]["wl2cal_time_diff"] = td_wl2cal.total_seconds()
    td_dr2wl = new_triplets[i]["dr2wl_time_diff"]
    new_triplets[i]["dr2wl_time_diff"] = td_dr2wl.total_seconds()

with open(dataset_dir / 'wl_dr_cal_matchings_new_max4h_aligned.json', 'w') as fp:
    json.dump(new_triplets, fp, indent=4)